In [4]:
import pandas as pd
import numpy as np
from pycaret.classification import *
import warnings
import os
warnings.filterwarnings('ignore')
import time
from datetime import datetime

# Configurações para otimizar performance no Mac M1
os.environ['OMP_NUM_THREADS'] = '8'  # Usar 8 threads
os.environ['MKL_NUM_THREADS'] = '8'

# Configurar ambiente
print("🚀 INICIANDO EXPERIMENTO SISTEMÁTICO PYCARET")
print("=" * 60)

# Carregar APENAS dados de treino (teste será usado só na validação final)
train_path = "/Users/marcelosilva/Desktop/projectOne/6/B- Binary Target DS DT/DatasetTrain.csv"

df_train = pd.read_csv(train_path)

print(f"✅ Dados carregados - Train: {df_train.shape}")
print(f"📝 Dataset de teste será preservado para validação final")

# Verificar se o target existe e suas características
target_col = 'status_nutricional_who'
if target_col in df_train.columns:
    print(f"✅ Target '{target_col}' encontrado!")
    print(f"   Distribuição: {df_train[target_col].value_counts().to_dict()}")
    print(f"   Tipo: {df_train[target_col].dtype}")
    print(f"   Valores únicos: {sorted(df_train[target_col].unique())}")
else:
    print(f"❌ ERRO: Target '{target_col}' NÃO encontrado!")
    print(f"   Colunas disponíveis: {list(df_train.columns)}")
    raise ValueError(f"Target column '{target_col}' not found in dataset")

# Remover colunas não desejadas
exclude_cols = ['id_anon', 'vd_zimc']
df_train_clean = df_train.drop(columns=exclude_cols, errors='ignore')

print(f"✅ Features disponíveis: {len(df_train_clean.columns)-1}")

# Definir conjuntos de features baseados na análise SHAP
feature_sets = {
    # Progressão por quantidade
    'top_1': ['imc_pre_gravidico'],
    'top_2': ['imc_pre_gravidico', 'k06_peso_engravidar'],
    'top_3': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso'],
    'top_4': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final'],
    'top_5': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 'imc_final_gravidico'],
    'top_8': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 
              'imc_final_gravidico', 'peso_concordancia', 'j03_cor', 'indice_ponderal'],
    'top_10': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 
               'imc_final_gravidico', 'peso_concordancia', 'j03_cor', 'indice_ponderal',
               'delta_imc', 'ganho_relativo'],
    'top_15': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 
               'imc_final_gravidico', 'peso_concordancia', 'j03_cor', 'indice_ponderal',
               'delta_imc', 'ganho_relativo', 'velocidade_ganho_semanal', 'quilos_por_imc_pre',
               'k05_prenatal_consultas', 'cobertura_prenatal', 'd01_cor'],
    
    # Grupos temáticos
    'antropometrico': ['imc_pre_gravidico', 'k06_peso_engravidar', 'k07_peso_final', 'h02_peso',
                      'imc_final_gravidico', 'h03_altura', 'indice_ponderal'],
    'demografico': ['j03_cor', 'd01_cor', 'b02_sexo', 'h04_parto'],
    'prenatal': ['k05_prenatal_consultas', 'k04_prenatal_semanas', 'cobertura_prenatal',
                'consultas_por_semana', 'adequacao_prenatal'],
    'gestacional': ['h01_semanas_gravidez', 'def_idade_gest', 'k12_tempo_horas', 'k18_somente_dias'],
    'peso_ganho': ['k08_quilos', 'delta_peso_absoluto_materno', 'delta_peso_relativo_materno',
                  'ganho_relativo', 'velocidade_ganho_semanal', 'ganho_por_parto'],
    
    # Combinações estratégicas
    'antropo_demo': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'j03_cor', 'd01_cor'],
    'antropo_prenatal': ['imc_pre_gravidico', 'k06_peso_engravidar', 'k05_prenatal_consultas', 
                        'cobertura_prenatal', 'h02_peso'],
    'core_mix': ['imc_pre_gravidico', 'j03_cor', 'k05_prenatal_consultas', 'h02_peso', 'h01_semanas_gravidez'],
    'peso_focused': ['imc_pre_gravidico', 'k06_peso_engravidar', 'k07_peso_final', 'h02_peso', 
                    'peso_concordancia', 'classificacao_peso'],
    'high_corr': ['k06_peso_engravidar', 'k07_peso_final', 'imc_pre_gravidico', 'imc_final_gravidico'],
    'balanced': ['imc_pre_gravidico', 'j03_cor', 'h02_peso', 'k05_prenatal_consultas', 
                'h01_semanas_gravidez', 'h04_parto', 'k08_quilos'],
    
    # Features completas
    'all_features': [col for col in df_train_clean.columns if col != 'status_nutricional_who']
}

# Algoritmos para testar
algorithms = ['rf', 'et', 'lightgbm', 'xgboost', 'catboost', 'dt', 'nb', 'svm', 'lr', 'knn', 
              'ada', 'gbc', 'lda', 'qda', 'ridge']

print(f"✅ Configurado {len(feature_sets)} conjuntos de features")
print(f"✅ Configurado {len(algorithms)} algoritmos")
print(f"✅ Total de experimentos: {len(feature_sets) * len(algorithms)}")

# Resultado dos experimentos
results = []

# Contador de experimentos
experiment_count = 0
total_experiments = len(feature_sets) * len(algorithms)

start_time = time.time()

print(f"\n🧪 INICIANDO {total_experiments} EXPERIMENTOS...")
print("=" * 60)

# Loop principal de experimentos
for feature_set_name, features in feature_sets.items():
    print(f"\n📊 Testando conjunto: {feature_set_name} ({len(features)} features)")
    
    # Verificar se todas as features existem
    available_features = [f for f in features if f in df_train_clean.columns]
    if len(available_features) != len(features):
        missing = set(features) - set(available_features)
        print(f"⚠️  Features não encontradas: {missing}")
        features = available_features
    
    # Preparar dataset para este conjunto de features
    train_subset = df_train_clean[features + ['status_nutricional_who']].copy()
    
    # Setup do PyCaret para este conjunto de features
    try:
        clf = setup(data=train_subset, 
                   target='status_nutricional_who',
                   session_id=123,
                   train_size=0.8,
                   fold_strategy='stratifiedkfold',
                   fold=5,
                   fix_imbalance=True,  # ← CORREÇÃO PRINCIPAL: Balancear dataset
                   verbose=False)
        
        # Testar cada algoritmo
        for algorithm in algorithms:
            experiment_count += 1
            
            try:
                start_alg_time = time.time()
                
                # Criar e treinar modelo com class balancing
                if algorithm in ['rf', 'et', 'xgboost', 'lightgbm', 'catboost', 'gbc', 'ada']:
                    # Tree-based models: usar class_weight='balanced'
                    model = create_model(algorithm, verbose=False, class_weight='balanced')
                elif algorithm in ['svm', 'lr']:
                    # Linear models: usar class_weight='balanced'
                    model = create_model(algorithm, verbose=False, class_weight='balanced')
                else:
                    # Outros modelos: usar sem class_weight
                    model = create_model(algorithm, verbose=False)
                
                # Avaliar modelo
                model_results = pull()
                
                end_alg_time = time.time()
                training_time = end_alg_time - start_alg_time
                
                # Extrair métricas principais (PyCaret 3.2.0)
                try:
                    # Pegar a última linha (média dos folds)
                    last_row = model_results.iloc[-1]
                    
                    # PyCaret 3.2.0 usa estes nomes de colunas
                    accuracy = float(last_row.get('Accuracy', 0))
                    precision = float(last_row.get('Prec.', 0))
                    recall = float(last_row.get('Recall', 0))
                    f1 = float(last_row.get('F1', 0))
                    auc = float(last_row.get('AUC', 0))
                    
                except Exception as metric_error:
                    accuracy = precision = recall = f1 = auc = 0
                
                # Armazenar resultado
                results.append({
                    'experiment_id': experiment_count,
                    'feature_set': feature_set_name,
                    'n_features': len(features),
                    'algorithm': algorithm,
                    'accuracy': float(accuracy),
                    'precision': float(precision),
                    'recall': float(recall),
                    'f1': float(f1),
                    'auc': float(auc),
                    'training_time': training_time,
                    'features_used': ', '.join(features)
                })
                
                print(f"✅ {experiment_count:3d}/{total_experiments} | {feature_set_name:12s} | {algorithm:8s} | Acc: {accuracy:.3f} | F1: {f1:.3f}")
                
            except Exception as e:
                results.append({
                    'experiment_id': experiment_count,
                    'feature_set': feature_set_name,
                    'n_features': len(features),
                    'algorithm': algorithm,
                    'accuracy': 0,
                    'precision': 0,
                    'recall': 0,
                    'f1': 0,
                    'auc': 0,
                    'training_time': 0,
                    'features_used': ', '.join(features),
                    'error': str(e)[:100]  # Limitar tamanho do erro
                })
    
    except Exception as e:
        print(f"❌ Setup erro: {feature_set_name} - {str(e)[:50]}")
        # Pular todos os algoritmos deste conjunto se setup falhar
        for algorithm in algorithms:
            experiment_count += 1

# Criar DataFrame com resultados
results_df = pd.DataFrame(results)

# Calcular tempo total
total_time = time.time() - start_time
print(f"\n⏱️  EXPERIMENTOS CONCLUÍDOS em {total_time/60:.1f} minutos")
print("=" * 60)

# Análise dos resultados
print(f"\n📈 ANÁLISE DOS RESULTADOS:")
print(f"Total de experimentos realizados: {len(results_df)}")

# Verificar se o DataFrame não está vazio e tem as colunas esperadas
if len(results_df) == 0:
    print("❌ ERRO: Nenhum resultado foi coletado!")
    print("Possíveis causas:")
    print("- Erro no setup do PyCaret")
    print("- Features não encontradas")
    print("- Problema com o target")
    exit()

# Verificar colunas do DataFrame
print(f"Colunas disponíveis: {list(results_df.columns)}")

# Filtrar apenas sucessos de forma segura
if 'accuracy' in results_df.columns:
    success_df = results_df[results_df['accuracy'] > 0].copy()
    print(f"Experimentos com sucesso: {len(success_df)}")
else:
    print("❌ ERRO: Coluna 'accuracy' não encontrada!")
    print("Verificando resultados disponíveis:")
    print(results_df.head() if len(results_df) > 0 else "DataFrame vazio")
    success_df = pd.DataFrame()  # DataFrame vazio para não quebrar o código

if len(success_df) > 0:
    # TOP 3 ALGORITMOS (por F1-Score média - melhor para dados desbalanceados)
    print(f"\n🏆 TOP 3 ALGORITMOS (por F1-Score média):")
    top_algorithms = success_df.groupby('algorithm').agg({
        'accuracy': 'mean',
        'f1': 'mean',
        'auc': 'mean',
        'training_time': 'mean'
    }).sort_values('f1', ascending=False).head(3)  # ← Ordenar por F1 em vez de accuracy
    
    for i, (alg, stats) in enumerate(top_algorithms.iterrows(), 1):
        print(f"{i}. {alg:12s} | F1: {stats['f1']:.4f} | Acc: {stats['accuracy']:.4f} | AUC: {stats['auc']:.4f}")
    
    # TOP 3 CONJUNTOS DE FEATURES (por F1-Score média)
    print(f"\n🎯 TOP 3 CONJUNTOS DE FEATURES (por F1-Score média):")
    top_features = success_df.groupby('feature_set').agg({
        'accuracy': 'mean',
        'f1': 'mean',
        'auc': 'mean',
        'n_features': 'first'
    }).sort_values('f1', ascending=False).head(3)  # ← Ordenar por F1 em vez de accuracy
    
    for i, (feat_set, stats) in enumerate(top_features.iterrows(), 1):
        print(f"{i}. {feat_set:15s} | F1: {stats['f1']:.4f} | Acc: {stats['accuracy']:.4f} | Features: {stats['n_features']}")
    
    # TOP 10 COMBINAÇÕES (algoritmo + features) por F1-Score
    print(f"\n🎖️  TOP 10 MELHORES COMBINAÇÕES (por F1-Score):")
    top_combinations = success_df.nlargest(10, 'f1')[['feature_set', 'algorithm', 'accuracy', 'f1', 'auc', 'n_features']]  # ← Ordenar por F1
    
    for i, (_, row) in enumerate(top_combinations.iterrows(), 1):
        print(f"{i:2d}. {row['algorithm']:10s} + {row['feature_set']:15s} | F1: {row['f1']:.4f} | Acc: {row['accuracy']:.4f}")

print(f"\n📊 RESUMO FINAL:")
if len(success_df) > 0:
    best_f1 = success_df['f1'].max()
    best_acc = success_df['accuracy'].max()
    print(f"🎯 Melhor F1-Score: {best_f1:.4f}")
    print(f"🎯 Melhor Accuracy: {best_acc:.4f}")
    print(f"📈 Baseline (classe majoritária): ~75.2%")
    print(f"📈 Melhoria esperada com balanceamento: Significativa!")

# Salvar resultados de forma segura
output_dir = "/Users/marcelosilva/Desktop/projectOne/6/D-Pycaret play"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Criar diretório se não existir
os.makedirs(output_dir, exist_ok=True)

# Salvar resultados completos sempre
results_df.to_csv(f"{output_dir}/pycaret_systematic_results_{timestamp}.csv", index=False)
print(f"\n💾 ARQUIVOS SALVOS:")
print(f"✅ pycaret_systematic_results_{timestamp}.csv - Todos os resultados")

# Salvar apenas sucessos se houver
if len(success_df) > 0:
    success_df.to_csv(f"{output_dir}/pycaret_successful_experiments_{timestamp}.csv", index=False)
    print(f"✅ pycaret_successful_experiments_{timestamp}.csv - Apenas sucessos")
    
    # Salvar top performers apenas se houver sucessos
    try:
        top_algorithms.to_csv(f"{output_dir}/top_3_algorithms_{timestamp}.csv")
        top_features.to_csv(f"{output_dir}/top_3_feature_sets_{timestamp}.csv")
        top_combinations.to_csv(f"{output_dir}/top_10_combinations_{timestamp}.csv", index=False)
        
        print(f"✅ top_3_algorithms_{timestamp}.csv - TOP 3 algoritmos")
        print(f"✅ top_3_feature_sets_{timestamp}.csv - TOP 3 conjuntos features")
        print(f"✅ top_10_combinations_{timestamp}.csv - TOP 10 combinações")
    except Exception as save_error:
        print(f"⚠️  Erro ao salvar rankings: {str(save_error)}")
else:
    print("⚠️  Nenhum experimento bem-sucedido para salvar rankings")

print(f"\n🎯 PRÓXIMOS PASSOS RECOMENDADOS:")
print(f"1. Analisar os TOP 3 algoritmos identificados")
print(f"2. Fazer hyperparameter tuning dos melhores")
print(f"3. Treinar modelos finais com o DatasetTrain.csv completo")
print(f"4. VALIDAÇÃO FINAL no DatasetTest.csv (ainda intocado)")
print(f"5. Reportar performance real no dados nunca vistos")

print(f"\n📋 ESTRATÉGIA DE VALIDAÇÃO:")
print(f"✅ Experimentos: Cross-validation no DatasetTrain.csv")
print(f"✅ Tuning: Cross-validation no DatasetTrain.csv") 
print(f"✅ Validação final: DatasetTest.csv (dados nunca vistos)")
print(f"✅ Prevenção de overfitting: datasets completamente separados")

print(f"\n🏁 EXPERIMENTO SISTEMÁTICO CONCLUÍDO!")

🚀 INICIANDO EXPERIMENTO SISTEMÁTICO PYCARET
✅ Dados carregados - Train: (3429, 39)
📝 Dataset de teste será preservado para validação final
✅ Target 'status_nutricional_who' encontrado!
   Distribuição: {0: 2580, 1: 849}
   Tipo: int64
   Valores únicos: [0, 1]
✅ Features disponíveis: 36
✅ Configurado 20 conjuntos de features
✅ Configurado 15 algoritmos
✅ Total de experimentos: 300

🧪 INICIANDO 300 EXPERIMENTOS...

📊 Testando conjunto: top_1 (1 features)
✅   1/300 | top_1        | rf       | Acc: 0.026 | F1: 0.030
✅   2/300 | top_1        | et       | Acc: 0.020 | F1: 0.022
✅   3/300 | top_1        | lightgbm | Acc: 0.026 | F1: 0.009
✅   4/300 | top_1        | xgboost  | Acc: 0.026 | F1: 0.013
✅   6/300 | top_1        | dt       | Acc: 0.022 | F1: 0.021
✅   7/300 | top_1        | nb       | Acc: 0.028 | F1: 0.014
✅   8/300 | top_1        | svm      | Acc: 0.230 | F1: 0.187
✅   9/300 | top_1        | lr       | Acc: 0.027 | F1: 0.018
✅  10/300 | top_1        | knn      | Acc: 0.024 | F1: